## Aannames
#### 1. LT-data en event confidence uitsluiten

Long-term data: akkoord, maar met nuancering

Het uitsluiten van LT-data als target is methodologisch de correcte beslissing, en ze is goed te verantwoorden. Hieronder de academische redenering.

Academische formulering voor de thesis:

"De short-term (ST) en long-term (LT) bezettingsdata worden in dit onderzoek gescheiden gemodelleerd. De LT-data, afkomstig van abonnementhouders met habitueel en nagenoeg constant rijgedrag, vertoont structureel andere variantiepatronen dan het transiente ST-gebruik dat centraal staat in de onderzoeksvragen (Lira et al., 2021). De target-variabele van het predictiemodel is bijgevolg uitsluitend de ST-bezettingsgraad (occupancy_rate uit MAD_shortterm.parquet). LT-bezetting wordt niet als target opgenomen, maar kan in nb08 worden geëvalueerd als contextuele feature voor parkings waar het empirisch is vastgesteld dat LT-druk de beschikbare ST-ruimte structureel beperkt (P Hoogstraat, P Komet — zie nb07b, Sectie 2.3)."

Kanttekening die je niet moet vergeten: voor P Hoogstraat en P Komet geldt een ceiling effect: LT neemt respectievelijk ~72% en ~72% van de capaciteit in. De ST-bezettingsgraad is er fundamenteel beperkt in variabiliteit (maximum bereikbaar is ~28%). Dit verlaagt de theoretische voorspelbaarheid voor die parkings. Vermeld dit expliciet als een per-parking data-limitatie — niet als een reden om die parkings uit te sluiten, maar als context bij het interpreteren van hogere MAPE-waarden die je er waarschijnlijk zal zien in nb11.

#### 2. Event confidence: gedeeltelijk akkoord — nuance is hier essentieel

Volledig uitsluiten van alle event-features is een valide maar onomkeerbare beslissing. De genuanceerde academische positie is sterker:

Optie A — Volledig uitsluiten (wat jij overweegt):

"Gegeven dat 53,5% van de event-observaties (185/346) een data_confidence = 'estimated' heeft — wat betekent dat tijdstip, duur en/of schaal niet empirisch zijn geverifieerd — wordt geconcludeerd dat de event-feature-set als geheel niet voldoet aan de meetbetrouwbaarheidsnormen die een ML-model vereist voor stabiele parameterschatting. Event-features worden bijgevolg niet opgenomen in de feature engineering pipeline (nb08). Dit is een expliciete databegrenzingsbeslissing: de bevindingen van H-S4 en H-E3 blijven relevant als exploratieve observaties, maar zijn onvoldoende robuust om te vertalen naar predictieve features."

Optie B — Alleen verified events behouden (methodologisch sterker):
Behoud de ~161 geverifieerde event-dagen als binaire dummies. De cascade-features (hours_to_next_event) worden dan alleen berekend op basis van verified events. Dit is wetenschappelijk eerlijker én haalt meer signaal eruit.

Mijn aanbeveling: Als je H-E3 (cascade) al bevestigd hebt, gooi je potentieel krachtige features weg bij Optie A. Optie B is methodologisch verdedigbaarder. Maar als je de administratieve last van event-verificatie te groot vindt voor de resterende thesis-tijd, dan is Optie A verdedigbaar mits je het expliciet rapporteert als bovenstaand.


#### 3. Waar en wanneer rapporteer je de 2020-gevoeligheidsanalyse en de lag-leakage beslissing?

2020-gevoeligheidsanalyse

Wanneer: In nb09 (baseline modellen), onmiddellijk na de eerste modeltraining.

Hoe: Train hetzelfde baseline model (bijv. XGBoost of Random Forest) twee keer:

Run A: trainingsdata = 2020 + 2023 + 2024

Run B: trainingsdata = 2023 + 2024 only

Valideer beide runs op 2024 (temporele cross-validatie) via MAPE of RMSE. Als Run B significant beter valideert, is 2020-data contraproductief en gebruik je alleen 2023+2024 als definitieve trainingsset.

Academische formulering voor de thesis (methodologie-sectie):

"Gezien de structureel afwijkende patronen in de COVID-periode 2020 (H-T4, r_struct = −0.956 voor centrumtier), werd een gevoeligheidsanalyse uitgevoerd waarbij het model werd getraind met (a) 2020 + 2023 + 2024 en (b) 2023 + 2024 uitsluitend. De validatieprestaties op het jaar 2024 bepaalden de definitieve keuze van de trainingsperiode."

#### 4.Lag-leakage: eerste week 2025 (occ_lag_168h)

Wanneer: In nb08 (feature engineering), in de cel die lag-features construeert.

Wat documenteren: Bij de constructie van occ_lag_168h voor de holdout (2025) zijn de eerste 168 uur (= week 1 van januari 2025) per definitie afhankelijk van trainingsdata (december 2024). Dit is correct en geen leakage. Maar vermeld expliciet:

Dat de lag-waarden voor de eerste week van de holdout zijn geïmporteerd vanuit de laatste week van de trainingsset

Dat er geen look-ahead leakage is (toekomstige bezetting wordt nooit gebruikt)

Dat je de eerste 168 uur van de holdout expliciet hebt gecontroleerd op NaN-waarden

Academische formulering (nb08, markdown-cel):

"Tijdreeksintegriteit: occ_lag_168h voor de holdoutperiode (2025) vereist bezettingswaarden uit de laatste week van de trainingsperiode (december 2024). Deze waarden zijn aaneengesloten beschikbaar en bevatten geen missing data na de kwaliteitsfilter. Er treedt geen temporele leakage op: geen enkel lag-feature bevat bezettingsinformatie uit de toekomst ten opzichte van het voorspelmoment. De eerste 168 observaties van de holdout worden gecontroleerd op NaN-completeness vóór modelinferentie."

#### 5. Steekproefomvang per event-type & daglengte-confound dubbelchecken

- Steekproefomvang per event-type
- Daglengte confound

# nb08 — Feature Engineering Pipeline

**Masterproef:** Tier-Stratified Occupancy Prediction and Scenario-Based Policy  
Simulation in Urban Parking Systems — A Case Study of Mechelen  
**Auteur:** Emile Van de Voorde  
**Status:** Feature engineering — input voor nb09 (baseline) t.e.m. nb13 (simulatie)  
**Vorige fase:** FASE02 EDA volledig afgerond (nb05 · nb06 · nb07 · nb07b)

---

## 1. Doel van deze notebook

Deze notebook construeert de definitieve feature-matrix die als input dient voor alle
modelleer- en simulatienotebooks (nb09–nb13). De feature-keuzes zijn empirisch
onderbouwd door 17 hypothesentoetsen in FASE02 en bijgesteld via vier post-hoc
analyses die na de initiële FASE02_Recap zijn uitgevoerd. Alle beslissingen worden
hieronder gedocumenteerd met motivatie en verwijzing naar de bronanalyse.

---

## 2. Databeslissingen vóór feature constructie

### 2.1 Target-variabele: uitsluitend short-term (ST) bezetting

De target-variabele is de short-term bezettingsgraad (`occupancy_rate` uit
`MAD_shortterm.parquet`). Long-term (LT) bezetting wordt **niet** als target
opgenomen op basis van de volgende motivatie (zie nb07b):

> *"De LT-bezetting, afkomstig van abonnementhouders met habitueel en nagenoeg
> constant rijgedrag, vertoont structureel andere variantiepatronen dan het
> transiente ST-gebruik dat centraal staat in de onderzoeksvragen (Lira et al.,
> 2021). LT-data is bijgevolg niet geschikt als modeldoelstelling."*

**Uitzondering — contextuele LT-feature:** Voor P Hoogstraat en P Komet is een
structureel LT-ceiling effect vastgesteld (LT-bezetting ~72% van capaciteit — zie
nb07b Sectie 2.3). Dit beperkt de maximaal bereikbare ST-variabiliteit voor deze
parkings. Een binaire dummy `high_lt_pressure` (P Hoogstraat = 1, P Komet = 1,
overige = 0) wordt opgenomen als lage-prioriteit contextuele feature. Hogere
MAPE-waarden voor deze twee parkings in nb11 zijn te verwachten en worden
expliciet gerapporteerd als per-parking limitatie.

### 2.2 Trainingsset: 2020-gevoeligheidsanalyse uitgesteld naar nb09

De trainingsset bevat voorlopig de volledige periode 2020 + 2023 + 2024
(129.837 rijen na kwaliteitsfilter). De COVID-periode 2020 vertoont structureel
afwijkende patronen ten opzichte van 2023/2024 (H-T4: r_struct = −0.956 voor
centrumtier; Δ = +18,6% voor vesten_of_rand — tegengesteld aan de verwachting).

Een formele gevoeligheidsanalyse (model getraind met vs. zonder 2020) wordt
uitgevoerd in **nb09**, onmiddellijk na de eerste baseline-modeltraining. De
uitkomst van die analyse bepaalt de definitieve trainingsperiode voor alle
vervolgmodellen. De `year_dummy` wordt in deze notebook **categorisch** gecodeerd
(2020 = 0, 2023/2024 = 1) — **niet ordinaal** — conform de bevindingen van H-T4.

### 2.3 Event-features: selectieve inclusie op basis van steekproefomvang

De event-data bevat een `data_confidence`-kolom waarbij 53,5% van de event-rijen
de waarde `'estimated'` heeft. Een generieke `is_event_day`-dummy maskeert
bovendien tegengestelde tier-effecten per event-type (H-S4, Wilcoxon matched pairs
analyse nb07). Op basis van steekproefgrootte en effectgrootte worden de volgende
beslissingen genomen:

| Event-type  | n event-days | Effect (Wilcoxon r) | Beslissing nb08        |
|-------------|:------------:|:-------------------:|------------------------|
| Kermis      | ~250         | 0.30–0.37 medium    | ✅ Opnemen — enkelvoudig |
| Voetbal     | ~90          | 0.12–0.22 klein     | ✅ Opnemen — met `× tier` |
| Festival    | ~15          | 0.37–0.40 medium    | ✅ Opnemen — met `× tier` |
| Carnaval    | ~35          | 0.06–0.24 klein     | 🟡 Opnemen — met `× tier`, lage prioriteit |
| Processie   | ~12          | 0.24–0.32 medium    | 🟡 Opnemen — enkelvoudig, lage prioriteit |

> ⚠ **Let op:** `event_label` bestaat niet als kolom in `df_train`. Het event-type
> wordt in Cel 03 van deze notebook afgeleid via keyword-matching op
> `event_names_combined` (zie classificatielogica aldaar). De `is_event_day`
> binaire kolom is **niet** bruikbaar als vervanging — die maskeert tegengestelde
> tier-effecten.

### 2.4 Lag-feature tijdreeksintegriteit (geen data leakage)

De lag-features `occ_lag_1h`, `occ_lag_24h` en `occ_lag_168h` vereisen
bezettingswaarden uit het verleden. Voor de eerste 168 uur van de holdoutperiode
(week 1 van januari 2025) zijn de lag-168h-waarden afkomstig uit de laatste week
van de trainingsperiode (december 2024). Deze waarden zijn aaneengesloten
beschikbaar en bevatten geen missing data na de kwaliteitsfilter. Er treedt geen
temporele leakage op: geen enkel lag-feature bevat bezettingsinformatie uit de
toekomst ten opzichte van het voorspelmoment. De eerste 168 observaties van de
holdout worden expliciet gecontroleerd op NaN-volledigheid in Cel 04.

---

## 3. Herziene master feature-shortlist

De onderstaande shortlist integreert alle aanpassingen ten opzichte van de
initiële FASE02_Recap, inclusief vier post-hoc correcties.

### 🔴 Prioriteit HOOG — opnemen zonder discussie

| Feature | Encoding | Basis | Wijziging t.o.v. FASE02_Recap |
|---|---|---|---|
| `occ_lag_1h` | Raw float [0,1] | H-T2: ACF lag-1 sig. | — |
| `occ_lag_24h` | Raw float [0,1] | H-T2: ACF lag-24 sig. | — |
| `occ_lag_168h` | Raw float [0,1] | H-T2: ACF lag-168 sig. | — |
| `hour_sin` + `hour_cos` | sin/cos(2π·h/24) | H-T1/T2 cyclische dagstructuur | — |
| `day_type_3` | One-hot (3 klassen) | H-T1 weekdag/zaterdag/zondag | — |
| `parking_id` | One-hot (10) of target encoding | H-S1: dominante ruimtelijke predictor | — |
| `tier_admin` | One-hot (2 klassen) | H-S1/S3: structureel niveauverschil | — |
| `cluster_data` | Binair (0/1) | Data-gedreven clustering nb06, silhouette=0.615 | **NIEUW** |
| `precip_bin` | 4 bins: droog/licht/matig/zwaar¹ | H-E1: drempelpatroon neerslag | — |
| `is_school_vacation` | Binair | H-E8: sterkste kalendereffect | — |
| `tier × is_school_vacation` | Interactieproduct | H-E8: Δr_rb=+0.117, p=0.000 | — |
| `year_dummy` | Categorisch (2020=0, anders=1) | H-T4: niet ordinaal | — |
| `is_kermis` | Binair | Wilcoxon r=0.30–0.37, n≈5.700 | Herbevestigd |
| `is_voetbal × tier` | Interactieproduct | Wilcoxon: tegengesteld tier-effect | Interactie verplicht |
| `is_festival × tier` | Interactieproduct | Wilcoxon r=0.37–0.40 (medium) | **OPGEWAARDEERD van verwijderd** |

¹ *Bingrenzen conform RMI-categorieën: droog = 0 mm/u, licht = 0–2 mm/u,
matig = 2–5 mm/u, zwaar > 5 mm/u.*

### 🟡 Prioriteit MEDIUM — testen in nb08, beslissing op basis van VIF/importance

| Feature | Encoding | Basis | Wijziging t.o.v. FASE02_Recap |
|---|---|---|---|
| `month_sin` + `month_cos` | sin/cos(2π·m/12) | Seizoenspatroon (H-T1) | — |
| `weekday_sin` + `weekday_cos` | sin/cos(2π·wd/7) | H-T2 wekelijkse cyclus | — |
| `wind_speed_ms` of dummy >10 m/s | Drempel of continu | H-E6: Δ mediaan = +5,3% | — |
| `hours_to_next_event` | Continu (min. 0) | H-E3: cascade pre-event sig. | — |
| `hours_since_last_event` | Continu (min. 0) | H-E3: cascade post-event sig. | — |
| `is_national_holiday × tier` | Interactieproduct | H-E4: Fischer z-bug → hertoetsen | Check na bugfix Fischer z |
| `is_carnaval × tier` | Interactieproduct | Wilcoxon r=0.06–0.24 | 🟡 Lage prioriteit, beperkte n |
| `is_processie` | Binair | Wilcoxon r=0.24–0.32, n=115–171 | 🟡 Lage prioriteit |

### 🟢 Prioriteit LAAG — optioneel / appendix

| Feature | Encoding | Basis | Wijziging t.o.v. FASE02_Recap |
|---|---|---|---|
| `high_lt_pressure` | Binair (P Hoogstraat, P Komet) | Ceiling effect nb07b | **NIEUW** |
| `sun_duration_min` | Gestandaardiseerd | H-E7 verworpen, collineair met temp | — |
| `total_capacity` | log-getransformeerd | H-S2: ρ=−0.11, redundant met parking_id | — |
| `n_concurrent_events` | Continu | H-E9: niet formeel bevestigd | — |

### ❌ Verwijderd t.o.v. FASE02_Recap

| Feature | Reden |
|---|---|
| `temp_c` (als zelfstandige feature) | Globale ρ = −0.013; Simpson's Paradox; seizoenspatroon volledig gevangen door `month_sin/cos` — post-hoc analyse nb07 |
| `temp_c × seizoen` interactie | Seizoen is de confound, niet de moderator — post-hoc analyse nb07 |
| Generieke `is_event_day` dummy | Maskeert tegengestelde tier-effecten (H-S4, Wilcoxon) |

---

## 4. Openstaande beslissingen — worden hier genomen vóór modellering

| Beslissing | Opties | Cel |
|---|---|---|
| Ruimtelijke encoding: `parking_id` one-hot vs. target encoding | One-hot (10) of mean-target encoding | Cel 05 |
| `cluster_data` gebruiken naast of i.p.v. `tier_admin` | Beide opnemen als kandidaten | Cel 05 |
| `precip_bin`-grenzen definitief vastleggen | RMI-categorieën (zie noot ¹) | Cel 06 |
| Lag-NaN-behandeling eerste week holdout | Forward-fill vanuit trainingsset | Cel 04 |
| Fischer z-implementatie gecontroleerd en gefixed | Zie wrapper-functie Cel 02 | Cel 02 |

---

## 5. Nog uit te voeren vóór nb09

- [ ] Gevoeligheidsanalyse 2020-data → uitkomst bepaalt definitieve trainingsset  
- [ ] Steekproefomvang per event-type verifiëren na `event_label_derived` constructie  
- [ ] NaN-check eerste 168 uur holdout voor `occ_lag_168h`  
- [ ] VIF-herberekening op volledige feature-set inclusief lag-features en
      interactietermen
